In [0]:
%pip install kagglehub

import kagglehub
import os
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType

# Tải dataset
path = kagglehub.dataset_download("lokeshparab/amazon-products-dataset")

# Tìm tất cả file CSV
csv_files = []
for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(root, f))

print(f"Found total: {len(csv_files)} files. Merging files altogether...")

# Định nghĩa Schema
schema = StructType([
    StructField("name", StringType(), True),
    StructField("main_category", StringType(), True),
    StructField("sub_category", StringType(), True),
    StructField("image", StringType(), True),
    StructField("link", StringType(), True),
    StructField("ratings", StringType(), True),
    StructField("no_of_ratings", StringType(), True),
    StructField("discount_price", StringType(), True),
    StructField("actual_price", StringType(), True)
])

# Đọc và gộp tất cả CSV
pdf_list = []
for i, file_path in enumerate(csv_files):
    try:
        temp_pdf = pd.read_csv(file_path, dtype=str, quotechar='"', on_bad_lines='skip', low_memory=False)
        for col in ["name", "main_category", "sub_category", "image", "link",
                    "ratings", "no_of_ratings", "discount_price", "actual_price"]:
            if col not in temp_pdf.columns:
                temp_pdf[col] = ""
        temp_pdf = temp_pdf[["name", "main_category", "sub_category", "image", "link",
                              "ratings", "no_of_ratings", "discount_price", "actual_price"]]
        pdf_list.append(temp_pdf)
        if (i + 1) % 20 == 0:
            print(f"Đã xử lý {i+1}/{len(csv_files)} files...")
    except Exception as e:
        print(f"Ignore error file {file_path}: {e}")

# Hợp nhất thành 1 Pandas DF
full_pdf = pd.concat(pdf_list, ignore_index=True).fillna("").astype(str)

# Ghi vào Delta Table (Bronze)
spark.sql("DROP TABLE IF EXISTS default.amazon_bronze")
df_bronze = spark.createDataFrame(full_pdf, schema=schema)
df_bronze.write.format("delta").mode("overwrite").saveAsTable("default.amazon_bronze")

print(f"Total number of rows of dataset: {df_bronze.count()}")
display(df_bronze.limit(10))

# Lưu file CSV gộp
full_pdf.to_csv("amazon_products_merged.csv", index=False, encoding='utf-8-sig')

In [0]:
from pyspark.sql import functions as F

print('EDA before cleaning')
print('=' * 50)
df_bronze = spark.table('default.amazon_bronze')

noise_price = df_bronze.withColumn(
    "is_noise",
    ~F.col("actual_price").rlike("^[0-9.,]+$")
).filter("is_noise = True").count()

noise_rating = df_bronze.withColumn(
    'is_noise',
    ~F.col('ratings').rlike('^[0-9]+(\\.[0-9]+)?$')
).filter('is_noise=True').count()

duplicate_names = df_bronze.groupby('name').count().filter('count>1').count()
total_rows = df_bronze.count()

print(f'Number of rows with format error: {noise_price:,} ({noise_price/total_rows*100:.2f}%)')
print(f'Number of rows with Rating error: {noise_rating:,} ({noise_rating/total_rows*100:.2f}%)')
print(f'Number of products with duplicate name: {duplicate_names}')

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

price_fallback = 1499.0
rating_fallback = 4.0

df_bronze = spark.table('default.amazon_bronze')

dedup_window = Window.partitionBy("name").orderBy(
    F.desc(F.length(F.col("actual_price"))),
    F.desc(F.length(F.col("ratings"))),
    F.col("name")
)

df_silver = df_bronze \
    .filter(F.col('name').isNotNull() & (F.col('name') != "")) \
    .withColumn("_rn", F.row_number().over(dedup_window)) \
    .filter(F.col("_rn") == 1).drop("_rn") \
    .withColumn('ratings_raw',
        F.expr("try_cast(regexp_extract(ratings, '([0-9]+\\.?[0-9]*)', 1) as float)")) \
    .withColumn('ratings_final',
        F.when((F.col("ratings_raw") >= 1.0) & (F.col("ratings_raw") <= 5.0),
               F.col("ratings_raw").cast("decimal(2,1)"))
         .otherwise(F.lit(rating_fallback).cast("decimal(2,1)"))) \
    .withColumn('price_raw',
        F.expr("try_cast(regexp_replace(actual_price, '[^0-9.]', '') as float)")) \
    .withColumn('price_final',
        F.when((F.col("price_raw") > 0) & (F.col("price_raw") < 1000000), F.col("price_raw"))
         .otherwise(F.lit(price_fallback))) \
    .withColumn('no_of_ratings',
        F.coalesce(F.expr("try_cast(regexp_replace(no_of_ratings, '[^0-9]', '') as int)"), F.lit(0))) \
    .withColumn('price_tier',
        F.when(F.col('price_raw') < 500, 'budget')
         .when(F.col('price_raw') < 5000, 'mid-range')
         .when(F.col('price_raw') >= 5000, 'premium')
         .otherwise('unknown price')) \
    .withColumn('rating_tier',
        F.when(F.col('ratings_final') >= 4.5, 'top rated')
         .when(F.col('ratings_final') >= 3.5, 'well rated')
         .otherwise('average rated')) \
    .withColumn('combined_text',
        F.trim(F.concat_ws(" | ",
            F.col("name"),
            F.coalesce(F.col("main_category"), F.lit("unknown")),
            F.coalesce(F.col("sub_category"), F.lit("unknown")),
            F.col('price_tier'),
            F.col('rating_tier')
        ))) \
    .select("name", "main_category", "sub_category", "image", "link",
            "ratings_final", "no_of_ratings", "price_final", "combined_text")

spark.sql('DROP TABLE IF EXISTS default.amazon_silver')
df_silver.write.format('delta').mode('overwrite').saveAsTable('default.amazon_silver')
print(f'SILVER DONE — Rows after cleaning: {df_silver.count()}')
display(df_silver.limit(10).toPandas())

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import seaborn as sns

df_silver = spark.table('default.amazon_silver')
total_rows = df_silver.count()
print(f'Total rows: {total_rows:,}')
print('=' * 55)

# --- Null/Missing Value Audit ---
null_check = []
for col_name in df_silver.columns:
    null_count = df_silver.filter(
        F.col(col_name).isNull() | (F.col(col_name).cast("string") == "")
    ).count()
    null_check.append({'column': col_name, 'null_count': null_count,
                       'null_pct': round(null_count / total_rows * 100, 2)})
null_df = spark.createDataFrame(null_check).orderBy(F.desc("null_count"))
display(null_df)

# --- Categorical Analysis ---
print('--- TOP 10 MAIN CATEGORIES')
display(df_silver.groupBy('main_category').count().orderBy(F.desc('count')).limit(10).toPandas())

print('--- TOP 10 SUB CATEGORIES')
display(df_silver.groupBy('sub_category').count().orderBy(F.desc('count')).limit(10).toPandas())

# --- Numerical Analysis ---
print('DESCRIPTIVE ANALYSIS (RATINGS & PRICE)')
display(df_silver.select('ratings_final', 'price_final').summary().toPandas())

# --- Price Bucketing ---
df_price_bucket = df_silver.withColumn("price_range",
    F.when(F.col("price_final") < 500, "< 500")
     .when((F.col("price_final") >= 500) & (F.col("price_final") < 2000), "500 - 2000")
     .when((F.col("price_final") >= 2000) & (F.col("price_final") < 10000), "2000 - 10000")
     .otherwise("> 10000"))
print('PRICE BUCKETING')
display(df_price_bucket.groupBy('price_range').count().toPandas())

# --- Text Length ---
df_text_stats = df_silver.withColumn('text_length', F.length('combined_text'))
print('TEXT LENGTH')
display(df_text_stats.select('text_length').summary().toPandas())

# --- Correlation Analysis ---
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=['ratings_final', 'price_final'], outputCol='features')
df_corr = df_silver \
    .withColumn('ratings_final', F.col('ratings_final').cast('float')) \
    .withColumn('price_final', F.col('price_final').cast('float'))
df_vector = assembler.transform(df_corr)
matrix = Correlation.corr(df_vector, 'features').collect()[0][0]
corr = matrix.toArray().tolist()[0][1]
strength = 'weak' if abs(corr) < 0.3 else 'moderate' if abs(corr) < 0.6 else 'strong'
direction = 'positive' if corr > 0 else 'negative'
print(f'Correlation between Price and Ratings: {corr:.4f} → {strength} {direction} correlation')

# --- Visualizations ---
sns.set_theme(style="whitegrid")
cat_data     = df_silver.groupBy('main_category').count().orderBy(F.desc('count')).limit(10).toPandas()
price_data   = df_price_bucket.groupBy('price_range').count().toPandas()
rating_dist  = df_silver.groupBy('ratings_final').count().orderBy('ratings_final').toPandas()

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(22, 7))

sns.barplot(data=cat_data, x='count', y='main_category', palette='viridis', ax=ax1)
ax1.set_title('Top 10 Main Categories', fontsize=14, fontweight='bold')
ax1.set_xlabel('Product Count'); ax1.set_ylabel('')

ax2.pie(price_data['count'], labels=price_data['price_range'], autopct='%1.1f%%',
        startangle=140, colors=sns.color_palette('pastel')[0:4],
        explode=[0.05]*len(price_data))
ax2.set_title('Price Range Distribution', fontsize=14, fontweight='bold')

sns.lineplot(data=rating_dist, x='ratings_final', y='count', marker='o', color='#e74c3c', ax=ax3)
ax3.fill_between(rating_dist['ratings_final'].astype(float), rating_dist['count'],
                 color='#e74c3c', alpha=0.2)
ax3.set_title('Ratings Distribution Trend', fontsize=14, fontweight='bold')
ax3.set_xlabel('Rating Score'); ax3.set_ylabel('Number of Products')

plt.tight_layout()
plt.show()

In [0]:
%pip install sentence-transformers

import pandas as pd
import os
from sentence_transformers import SentenceTransformer
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, FloatType
from pyspark.sql.window import Window

# --- Pandas UDF tạo embeddings ---
_model_holder = None

@pandas_udf(ArrayType(FloatType()))
def compute_embeddings_safe(texts: pd.Series) -> pd.Series:
    global _model_holder
    if _model_holder is None:
        os.environ['HF_HOME'] = '/tmp/huggingface'
        os.environ['SENTENCE_TRANSFORMERS_HOME'] = '/tmp/sentence_transformers'
        _model_holder = SentenceTransformer('all-MiniLM-L6-v2', cache_folder='/tmp/models')
    embeddings = _model_holder.encode(
        texts.tolist(), batch_size=16, show_progress_bar=False, convert_to_numpy=True
    )
    return pd.Series(embeddings.tolist())

# --- Lấy top sản phẩm mỗi category ---
MAX_PER_CATEGORY = 500
ROWS_PER_PARTITION = 50
num_partitions = max(1, min(600, 30000 // ROWS_PER_PARTITION))

windowSpec = Window.partitionBy("main_category").orderBy(
    F.desc(F.col("ratings_final") * F.log1p(F.col("no_of_ratings").cast("float")))
)
df_top_per_cat = spark.table("default.amazon_silver") \
    .withColumn("rank", F.row_number().over(windowSpec)) \
    .filter(F.col("rank") <= MAX_PER_CATEGORY).drop("rank")

df_random = spark.table("default.amazon_silver") \
    .join(df_top_per_cat.select("name"), on="name", how="left_anti") \
    .sample(False, 0.05, seed=42)

gold_dedup_window = Window.partitionBy("name").orderBy(
    F.desc(F.col("ratings_final") * F.log1p(F.col("no_of_ratings").cast("float")))
)
df_gold_input = df_top_per_cat.unionByName(df_random, allowMissingColumns=True) \
    .withColumn("_rn", F.row_number().over(gold_dedup_window)) \
    .filter(F.col("_rn") == 1).drop("_rn") \
    .orderBy(F.col("name")).limit(30000).repartition(num_partitions)

# --- Tính embeddings và ghi Gold table ---
df_gold = df_gold_input.withColumn("vector", compute_embeddings_safe(F.col("combined_text")))

spark.sql("DROP TABLE IF EXISTS default.amazon_gold")
df_gold.write.format("delta").mode("overwrite").saveAsTable("default.amazon_gold")

# --- Thống kê pipeline ---
count_bronze = spark.table("default.amazon_bronze").count()
count_silver = spark.table("default.amazon_silver").count()
count_gold   = spark.table("default.amazon_gold").count()
print(f"Pipeline: Bronze ({count_bronze:,}) → Silver ({count_silver:,}) → Gold ({count_gold:,})")
display(spark.table("default.amazon_gold").groupBy("main_category").count().orderBy(F.desc("count")))

In [0]:
%pip install faiss-cpu

import faiss
import pandas as pd
import numpy as np
import os
import time
from sentence_transformers import SentenceTransformer
from typing import Tuple, Dict

class ProductSearchEngine:
    def __init__(self):
        self.model = None
        self.index = None
        self.data = None
        self._initialized = False
        self._index_path = '/tmp/faiss_product_index.bin'

    def initialize(self, data_table: str = 'default.amazon_gold'):
        if self._initialized:
            return
        os.environ['HF_HOME'] = '/tmp/huggingface'
        self.model = SentenceTransformer('all-MiniLM-L6-v2', cache_folder='/tmp/models')
        self.data = spark.table(data_table).toPandas()
        vectors = np.array(self.data['vector'].tolist()).astype('float32')
        faiss.normalize_L2(vectors)
        self.index = faiss.IndexFlatIP(vectors.shape[1])
        self.index.add(vectors)
        self._initialized = True
        print(f'Search Engine Initialized: {self.index.ntotal} products')

    def save_index(self, path: str = None):
        if not self._initialized:
            raise RuntimeError('Search engine is not initialized')
        save_path = path or self._index_path
        try:
            faiss.write_index(self.index, save_path)
            print(f'FAISS index saved to: {save_path}')
            return save_path
        except Exception as e:
            print(f'Could not save index: {e}')
            return None

    def load_index(self, path: str = None):
        load_path = path or self._index_path
        try:
            self.index = faiss.read_index(load_path)
            print(f'FAISS index loaded from: {load_path}')
            print(f'Vectors: {self.index.ntotal}, Dimensions: {self.index.d}')
            return True
        except Exception as e:
            print(f'Could not load index from {load_path}: {e}')
            return False

    def search(self, query: str, category: str = 'All', k: int = 50):
        if not self._initialized:
            raise RuntimeError('Search engine not initialized. Call initialize() first')
        start = time.time()
        q_vec = self.model.encode([query]).astype('float32')
        q_vec = np.ascontiguousarray(q_vec)
        faiss.normalize_L2(q_vec)
        search_k = min(k * 20, self.index.ntotal) if (category and category != 'All') else k
        scores, indices = self.index.search(q_vec, search_k)
        results = self.data.iloc[indices[0]].copy()
        results['similarity'] = scores[0]
        if category and category != 'All':
            results = results[results['main_category'] == category]
        results = results.head(k).drop(columns=['vector'], errors='ignore')
        total_latency = (time.time() - start) * 1000
        metadata = {
            'query': query,
            'category': category,
            'latency_ms': total_latency,
            'num_results': len(results),
            'index_size': self.index.ntotal
        }
        return results, metadata


# ── Khởi tạo Search Engine ──────────────────────────────────────────────────
try:
    if search_engine._initialized:
        print('✅ Search engine already initialized. Skipping.')
    else:
        raise AttributeError('Not initialized')

except (NameError, AttributeError):
    search_engine = ProductSearchEngine()

    # ① Thử load FAISS index đã lưu từ lần trước → tiết kiệm thời gian build lại
    index_loaded = search_engine.load_index()

    if index_loaded:
        # Index đã load nhưng data (pandas) vẫn cần load để search trả kết quả
        print('📦 Loading product data from Gold table...')
        os.environ['HF_HOME'] = '/tmp/huggingface'
        search_engine.model = SentenceTransformer('all-MiniLM-L6-v2', cache_folder='/tmp/models')
        search_engine.data = spark.table('default.amazon_gold').toPandas()
        search_engine._initialized = True
        print(f'✅ Search engine ready (from cache)!')
    else:
        # ② Không có cache → build từ đầu rồi lưu lại
        print('🔨 No cache found. Building index from scratch...')
        search_engine.initialize()
        search_engine.save_index()
        print('✅ Search engine ready (newly built)!')

    print(f'   {search_engine.index.ntotal:,} products indexed')
    print(f'   Use: results, meta = search_engine.search(query)')

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict

# Guard check: ensure search engine is ready
if 'search_engine' not in globals() or not search_engine._initialized:
    raise RuntimeError("❌ Search engine not initialized. Please run Cell 11 first.")

print("✅ Search engine detected and ready")

# Test Cases (FIXED: removed duplicate, fixed circular keywords)
test_cases = [
    {
        "query": "low priced device for online classes",
        "expected_keywords": ["laptop", "tablet", "webcam", "headphone", "lenovo", "asus"],
        "category": "Computing",
        "note": "Context test: 'online classes' thường gắn với laptop/tablet giá rẻ."
    },
    {
        "query": "protection for expensive smartphones",
        "expected_keywords": ["case", "cover", "screen", "guard", "tempered"],
        "category": "Accessories",
        "note": "Functional test: 'protection' tương đương với ốp lưng/kính cường lực."
    },
    {
        "query": "equipment for brewing coffee at home",
        "expected_keywords": ["maker", "machine", "grinder", "press", "filter"],
        "category": "Home & Kitchen",
        "note": "Concept test: 'brewing coffee' liên quan đến máy pha/máy xay cà phê."
    },
    {
        "query": "professional tools for vlogging",
        "expected_keywords": ["tripod", "ring", "light", "mic", "microphone", "gimbal"],
        "category": "Electronics",
        "note": "Inference test: 'vlogging' gợi ý các thiết bị hỗ trợ quay phim."
    },
    {
        "query": "apparel for winter sports",
        "expected_keywords": ["jacket", "thermal", "glove", "sweater", "hoodie"],
        "category": "Clothing",
        "note": "Category inference: 'winter sports' thường là áo khoác/đồ giữ nhiệt."
    },
    {
        "query": "storage solution for high res photos",
        "expected_keywords": ["hard", "drive", "ssd", "memory", "card", "sandisk"],
        "category": "Electronics",
        "note": "Problem-solving test: 'storage solution' là ổ cứng hoặc thẻ nhớ."
    },
    {
        "query": "cooking appliance for healthy oil free meals",
        "expected_keywords": ["air", "fryer", "steamer", "grill"],
        "category": "Home & Kitchen",
        "note": "Descriptive test: 'oil free' là đặc tính của nồi chiên không dầu."
    },
    {
        "query": "noise cancelling gear for travelers",
        "expected_keywords": ["headphone", "earbud", "sony", "bose", "silence"],
        "category": "Audio",
        "note": "Specific feature test: 'noise cancelling' là tính năng của tai nghe cao cấp."
    },
    {
        "query": "device to turn tv into smart tv",
        "expected_keywords": ["fire", "stick", "chromecast", "box", "mi"],
        "category": "Electronics",
        "note": "Use-case test: Mô tả công dụng thay vì gọi tên thiết bị."
    },
    {
        "query": "ergonomic gear for office workers",
        "expected_keywords": ["chair", "mouse", "keyboard", "stand", "support"],
        "category": "Furniture",
        "note": "Adjective test: 'ergonomic' thường đi với ghế hoặc bàn phím văn phòng."
    }
]

print(f"📋 Loaded {len(test_cases)} test queries across {len(set(tc['category'] for tc in test_cases))} categories")

# Keyword Search Baseline
def keyword_search(query_text: str, k: int = 10) -> pd.DataFrame:
    """Baseline: simple keyword matching"""
    keywords = query_text.lower().split()
    pdf = search_engine.data.copy()
    pdf['kw_score'] = pdf['name'].apply(
        lambda x: sum(1 for kw in keywords if kw in str(x).lower())
    )
    pdf = pdf.sort_values(['kw_score', 'ratings_final'], ascending=[False, False])
    return pdf.head(k)[['name', 'ratings_final', 'price_final', 'kw_score']]

# Evaluation Metrics
def precision_at_k(results_df: pd.DataFrame, expected_keywords: List[str], k: int = 5) -> float:
    """Calculate Precision@K"""
    top_k = results_df.head(k)
    hits = sum(
        1 for _, row in top_k.iterrows()
        if any(kw in str(row['name']).lower() for kw in expected_keywords)
    )
    return hits / k

def mean_reciprocal_rank(results_df: pd.DataFrame, expected_keywords: List[str]) -> float:
    """Calculate Mean Reciprocal Rank (MRR)"""
    for rank, (_, row) in enumerate(results_df.iterrows(), start=1):
        if any(kw in str(row['name']).lower() for kw in expected_keywords):
            return 1.0 / rank
    return 0.0

def evaluate_search_method(search_fn, test_cases: List[Dict], k: int = 5) -> pd.DataFrame:
    """Evaluate a search method across test cases"""
    results = []
    for case in test_cases:
        query = case['query']
        expected = case['expected_keywords']
        category = case['category']
        
        search_results = search_fn(query)
        
        p_at_k = precision_at_k(search_results, expected, k=k)
        mrr = mean_reciprocal_rank(search_results, expected)
        
        results.append({
            'query': query,
            'category': category,
            f'precision@{k}': p_at_k,
            'mrr': mrr
        })
    
    return pd.DataFrame(results)

print("✅ Evaluation functions ready")

# FIXED: Use search_engine.search() directly (no HTML rendering)
def semantic_search_eval(query: str) -> pd.DataFrame:
    results, _ = search_engine.search(query, k=10)
    return results[['name', 'ratings_final', 'price_final', 'similarity']]

# Run Evaluation
print("\n🔄 Running evaluation...\n")

K = 5
semantic_results = evaluate_search_method(semantic_search_eval, test_cases, k=K)
keyword_results = evaluate_search_method(keyword_search, test_cases, k=K)

semantic_results['method'] = 'Semantic Search (FAISS)'
keyword_results['method'] = 'Keyword Search (Baseline)'

comparison_df = pd.concat([semantic_results, keyword_results], ignore_index=True)

print("✅ Evaluation complete!\n")

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("="*80)
print("📊 EVALUATION RESULTS SUMMARY")
print("="*80)

# Overall metrics
for method in ['Semantic Search (FAISS)', 'Keyword Search (Baseline)']:
    method_data = comparison_df[comparison_df['method'] == method]
    avg_precision = method_data[f'precision@{K}'].mean()
    avg_mrr = method_data['mrr'].mean()
    
    print(f"\n{method}:")
    print(f"  • Average Precision@{K}: {avg_precision:.3f} ({avg_precision*100:.1f}%)")
    print(f"  • Average MRR: {avg_mrr:.3f}")

# Calculate improvement
semantic_p = semantic_results[f'precision@{K}'].mean()
keyword_p = keyword_results[f'precision@{K}'].mean()
improvement = ((semantic_p - keyword_p) / keyword_p) * 100

print(f"\n🚀 Semantic search outperforms keyword search by {improvement:+.1f}%")
print("="*80)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 10)

# ===== 1. PIVOT TABLE: SIDE-BY-SIDE COMPARISON =====
print("\n" + "="*100)
print("📊 DETAILED COMPARISON TABLE (Side-by-Side)")
print("="*100 + "\n")

# Create pivot for Precision@5
precision_pivot = comparison_df.pivot_table(
    index=['query', 'category'],
    columns='method',
    values='precision@5'
).reset_index()

# Create pivot for MRR
mrr_pivot = comparison_df.pivot_table(
    index=['query', 'category'],
    columns='method',
    values='mrr'
).reset_index()

# Merge both metrics
comparison_table = precision_pivot.merge(
    mrr_pivot,
    on=['query', 'category'],
    suffixes=('_precision', '_mrr')
)

# Rename columns for clarity
comparison_table.columns = [
    'Query', 'Category',
    'Semantic P@5', 'Keyword P@5',
    'Semantic MRR', 'Keyword MRR'
]

# Add delta columns
comparison_table['P@5 Δ'] = comparison_table['Semantic P@5'] - comparison_table['Keyword P@5']
comparison_table['MRR Δ'] = comparison_table['Semantic MRR'] - comparison_table['Keyword MRR']

# Format percentages
for col in ['Semantic P@5', 'Keyword P@5', 'P@5 Δ']:
    comparison_table[col] = comparison_table[col].apply(lambda x: f"{x*100:.0f}%")

for col in ['Semantic MRR', 'Keyword MRR', 'MRR Δ']:
    comparison_table[col] = comparison_table[col].apply(lambda x: f"{x:.3f}")

# Sort by category and delta
comparison_table = comparison_table.sort_values(['Category', 'Query'])

print("\n🔍 Legend: P@5 = Precision@5, MRR = Mean Reciprocal Rank, Δ = Improvement\n")
display(comparison_table)

# ===== 2. BAR CHART: AVERAGE METRICS COMPARISON =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Precision@5 comparison
avg_metrics = comparison_df.groupby('method').agg({
    'precision@5': 'mean',
    'mrr': 'mean'
}).reset_index()

colors = ['#2ecc71', '#e74c3c']
x_pos = np.arange(len(avg_metrics))

# Plot Precision@5
ax1.bar(x_pos, avg_metrics['precision@5'], color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(avg_metrics['method'], rotation=15, ha='right')
ax1.set_ylabel('Average Precision@5', fontsize=12, fontweight='bold')
ax1.set_title('Average Precision@5 Comparison', fontsize=14, fontweight='bold')
ax1.set_ylim([0, 1])
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(avg_metrics['precision@5']):
    ax1.text(i, v + 0.02, f'{v:.3f}\n({v*100:.1f}%)', ha='center', fontweight='bold', fontsize=11)

# Plot MRR
ax2.bar(x_pos, avg_metrics['mrr'], color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(avg_metrics['method'], rotation=15, ha='right')
ax2.set_ylabel('Average MRR', fontsize=12, fontweight='bold')
ax2.set_title('Average MRR Comparison', fontsize=14, fontweight='bold')
ax2.set_ylim([0, 1])
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(avg_metrics['mrr']):
    ax2.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

# ===== 3. HEATMAP: PER-QUERY PERFORMANCE =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Precision@5 heatmap
precision_matrix = comparison_df.pivot_table(
    index='query',
    columns='method',
    values='precision@5'
)

sns.heatmap(
    precision_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=0,
    vmax=1,
    cbar_kws={'label': 'Precision@5'},
    ax=ax1,
    linewidths=0.5
)
ax1.set_title('Precision@5 Heatmap (per Query)', fontsize=14, fontweight='bold')
ax1.set_xlabel('')
ax1.set_ylabel('Query', fontsize=11)

# MRR heatmap
mrr_matrix = comparison_df.pivot_table(
    index='query',
    columns='method',
    values='mrr'
)

sns.heatmap(
    mrr_matrix,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',
    vmin=0,
    vmax=1,
    cbar_kws={'label': 'MRR'},
    ax=ax2,
    linewidths=0.5
)
ax2.set_title('MRR Heatmap (per Query)', fontsize=14, fontweight='bold')
ax2.set_xlabel('')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

# ===== 4. CATEGORY-WISE COMPARISON =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

category_metrics = comparison_df.groupby(['category', 'method']).agg({
    'precision@5': 'mean',
    'mrr': 'mean'
}).reset_index()

categories = category_metrics['category'].unique()
x = np.arange(len(categories))
width = 0.35

# Precision@5 by category
semantic_p5 = category_metrics[category_metrics['method'] == 'Semantic Search (FAISS)']['precision@5'].values
keyword_p5 = category_metrics[category_metrics['method'] == 'Keyword Search (Baseline)']['precision@5'].values

ax1.bar(x - width/2, semantic_p5, width, label='Semantic', color='#2ecc71', alpha=0.8, edgecolor='black')
ax1.bar(x + width/2, keyword_p5, width, label='Keyword', color='#e74c3c', alpha=0.8, edgecolor='black')
ax1.set_xlabel('Category', fontsize=12, fontweight='bold')
ax1.set_ylabel('Average Precision@5', fontsize=12, fontweight='bold')
ax1.set_title('Precision@5 by Category', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(categories, rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim([0, 1.1])

# MRR by category
semantic_mrr = category_metrics[category_metrics['method'] == 'Semantic Search (FAISS)']['mrr'].values
keyword_mrr = category_metrics[category_metrics['method'] == 'Keyword Search (Baseline)']['mrr'].values

ax2.bar(x - width/2, semantic_mrr, width, label='Semantic', color='#2ecc71', alpha=0.8, edgecolor='black')
ax2.bar(x + width/2, keyword_mrr, width, label='Keyword', color='#e74c3c', alpha=0.8, edgecolor='black')
ax2.set_xlabel('Category', fontsize=12, fontweight='bold')
ax2.set_ylabel('Average MRR', fontsize=12, fontweight='bold')
ax2.set_title('MRR by Category', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(categories, rotation=45, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim([0, 1.1])

plt.tight_layout()
plt.show()

# ===== 5. WIN/LOSS SUMMARY =====
print("\n" + "="*100)
print("🏆 WIN/LOSS SUMMARY")
print("="*100 + "\n")

# Count wins per metric
precision_wins = (precision_matrix['Semantic Search (FAISS)'] > precision_matrix['Keyword Search (Baseline)']).sum()
mrr_wins = (mrr_matrix['Semantic Search (FAISS)'] > mrr_matrix['Keyword Search (Baseline)']).sum()

total_queries = len(test_cases)

print(f"📊 Out of {total_queries} test queries:\n")
print(f"  🟢 Semantic wins (Precision@5): {precision_wins}/{total_queries} ({precision_wins/total_queries*100:.0f}%)")
print(f"  🟢 Semantic wins (MRR):          {mrr_wins}/{total_queries} ({mrr_wins/total_queries*100:.0f}%)")
print(f"\n  🔴 Keyword wins (Precision@5):   {total_queries - precision_wins}/{total_queries} ({(total_queries - precision_wins)/total_queries*100:.0f}%)")
print(f"  🔴 Keyword wins (MRR):           {total_queries - mrr_wins}/{total_queries} ({(total_queries - mrr_wins)/total_queries*100:.0f}%)")
print("\n" + "="*100)



In [0]:
import time
import numpy as np
import faiss
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

d  = 384
nb = spark.table("default.amazon_gold").count()
xb = np.random.random((nb, d)).astype('float32')
xq = np.random.random((1, d)).astype('float32')
print(f"Benchmark trên {nb:,} bản ghi Gold, dim={d}")

# Brute-force (Numpy)
start = time.time()
distances = np.linalg.norm(xb - xq, axis=1)
indices   = np.argsort(distances)[:5]
bruteforce_time = (time.time() - start) * 1000

# FAISS
index = faiss.IndexFlatIP(d)
index.add(xb)
start = time.time()
D, I  = index.search(xq, 5)
faiss_time = (time.time() - start) * 1000

print(f"Brute-force: {bruteforce_time:.2f} ms")
print(f"FAISS:       {faiss_time:.2f} ms")
print(f"Improvement Ratio: {bruteforce_time / faiss_time:.1f}x")

# Scalability curve
sizes     = [1000, 5000, 10000, 29000, 50000, 100000]
latencies = []
for size in sizes:
    xtmp = np.random.random((size, d)).astype('float32')
    idx  = faiss.IndexFlatIP(d)
    idx.add(xtmp)
    start = time.time()
    idx.search(xq, 5)
    latencies.append((time.time() - start) * 1000)

plt.figure(figsize=(10, 5))
plt.plot(sizes, latencies, marker='o', linestyle='-', color='orange')
plt.axhline(y=50, color='r', linestyle='--', label='Threshold 50ms')
plt.title('Scalability Analysis: Search Latency vs. Dataset Size', fontweight='bold')
plt.xlabel('Number of Products'); plt.ylabel('Latency (ms)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [0]:
if 'search_engine' not in globals() or not search_engine._initialized:
    raise RuntimeError("❌ Hãy chạy cell khởi tạo SearchEngine trước!")

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ======================
# 1. KHỞI TẠO CÁC WIDGETS (UI)
# ======================
search_input = widgets.Text(
    placeholder='Search for product (VD: affordable gaming laptop)...', 
    layout=widgets.Layout(width='450px')
)

search_button = widgets.Button(
    description='🔍Search',
    button_style='primary',
    layout=widgets.Layout(width='130px', height='38px')
)

top_k_slider = widgets.IntSlider(
    value=10, min=5, max=20, step=1,
    description='Number of results:',
    layout=widgets.Layout(width='400px')
)

# Lấy danh sách category từ data của engine
categories = ["All"] + sorted([str(c) for c in search_engine.data['main_category'].unique() if c])
category_dropdown = widgets.Dropdown(
    options=categories,
    value='All',
    description='Category:',
    layout=widgets.Layout(width='320px')
)

output_area = widgets.Output()

# ======================
# 2. LOGIC XỬ LÝ TÌM KIẾM & RENDER HTML
# ======================
def run_search(query: str, top_k: int, category: str):
    if not query.strip():
        with output_area:
            clear_output()
            print("⚠️ Please input the query!")
        return

    # GỌI ENGINE MỚI (Có tham số category)
    results, meta = search_engine.search(query, k=top_k, category=category)

    if results.empty:
        with output_area:
            clear_output()
            print(f"❌ Cannot find related product in '{category}'")
        return

    # Chuẩn hóa dữ liệu để vẽ thanh Progress Bar
    results = results.copy()
    results['rating_norm'] = (results['ratings_final'].astype(float) - 1) / 4
    
    p_min, p_max = results['price_final'].min(), results['price_final'].max()
    # Công thức đảo ngược: Giá thấp hơn thì điểm "Value" cao hơn
    results['price_norm'] = 1 - (results['price_final'] - p_min) / (p_max - p_min + 1e-6)

    cards_html = ""
    for i, row in results.reset_index(drop=True).iterrows():
        rank = i + 1
        rank_icon = {1: "🥇", 2: "🥈", 3: "🥉"}.get(rank, f"#{rank}")
        
        full_name = str(row['name'])
        short_name = full_name[:100] + "..." if len(full_name) > 100 else full_name
        is_long = len(full_name) > 100

        sim_pct = float(row['similarity']) * 100
        rat_pct = max(0, min(100, float(row['rating_norm']) * 100))
        pri_pct = max(0, min(100, float(row['price_norm']) * 100))
        
        cat_label = str(row.get('main_category') or 'N/A')
        price_val = float(row['price_final'])
        rating_val = row['ratings_final']
        
        border_color = {1: "#FFD700", 2: "#A0A0A0", 3: "#CD7F32"}.get(rank, "#3B82F6")
        card_id = f"card_{rank}"

        # Toggle Button cho tên dài
        toggle_btn = f"""
            <span id="btn_{card_id}" onclick="
                var el = document.getElementById('full_{card_id}');
                if(el.style.display==='none') {{
                    el.style.display='block'; this.innerText='▲ Thu gọn';
                }} else {{
                    el.style.display='none'; this.innerText='▼ Xem thêm';
                }}
            " style="cursor:pointer; font-size:11px; color:#3B82F6; font-weight:600; margin-left:8px;">▼ Xem thêm</span>
        """ if is_long else ""

        cards_html += f"""
        <div style="background:white; border-left:6px solid {border_color}; border-radius:12px; 
                    padding:16px; margin-bottom:12px; box-shadow:0 2px 8px rgba(0,0,0,0.05); font-family:sans-serif;">
            <div style="display:flex; gap:12px;">
                <div style="font-size:22px; font-weight:bold; color:{border_color}; min-width:40px;">{rank_icon}</div>
                <div style="flex:1;">
                    <div style="font-size:14px; font-weight:700; color:#111827;">{short_name} {toggle_btn}</div>
                    <div id="full_{card_id}" style="display:none; font-size:13px; color:#4B5563; background:#F9FAFB; padding:8px; margin-top:5px; border-radius:6px;">{full_name}</div>
                    
                    <div style="display:flex; gap:8px; margin:10px 0;">
                        <span style="background:#FEF3C7; color:#92400E; padding:2px 8px; border-radius:4px; font-size:11px; font-weight:bold;">⭐ {rating_val}</span>
                        <span style="background:#ECFDF5; color:#065F46; padding:2px 8px; border-radius:4px; font-size:11px; font-weight:bold;">₹{price_val:,.0f}</span>
                        <span style="background:#EFF6FF; color:#1D4ED8; padding:2px 8px; border-radius:4px; font-size:11px; font-weight:bold;">📂 {cat_label}</span>
                    </div>

                    <div style="margin-top:10px;">
                        {_gen_bar("🎯 SIMILARITY", sim_pct, "linear-gradient(90deg,#3B82F6,#6366F1)")}
                        {_gen_bar("⭐ QUALITY", rat_pct, "#10B981")}
                        {_gen_bar("💰 VALUE", pri_pct, "#F59E0B")}
                    </div>
                </div>
            </div>
        </div>
        """

    header_html = f"""
    <div style="margin-bottom:15px; padding:10px; background:#F3F4F6; border-radius:8px; font-family:sans-serif; font-size:12px; color:#6B7280; display:flex; gap:20px;">
        <span>⚡ Latency: <b>{meta['latency_ms']:.1f}ms</b></span>
        <span>📦 Index: <b>{meta['index_size']:,} products</b></span>
        <span>🔍 Category: <b>{category}</b></span>
    </div>
    """

    with output_area:
        clear_output(wait=True)
        display(HTML(header_html + cards_html))

# Hàm phụ trợ tạo progress bar
def _gen_bar(label, pct, color):
    return f"""
    <div style="margin-bottom:5px;">
        <div style="display:flex; justify-content:space-between; font-size:10px; font-weight:bold; color:#6B7280; margin-bottom:2px;">
            <span>{label}</span><span>{pct:.1f}%</span>
        </div>
        <div style="background:#E5E7EB; height:5px; border-radius:3px; overflow:hidden;">
            <div style="width:{min(pct,100)}%; height:100%; background:{color};"></div>
        </div>
    </div>
    """
# Gán hàm phụ trợ vào local namespace để tiện dùng trong UI
import types
ui_helpers = types.SimpleNamespace(_gen_bar=_gen_bar)

# ======================
# 3. KẾT NỐI SỰ KIỆN & HIỂN THỊ
# ======================
def on_click_wrapper(b):
    run_search(search_input.value, top_k_slider.value, category_dropdown.value)

search_button.on_click(on_click_wrapper)

# Hiển thị khối điều khiển chuyên nghiệp
ui_box = widgets.VBox([
    widgets.HTML("<h2 style='color:#1D4ED8; font-family:sans-serif;'>Amazon Hybrid Search Engine</h2>"),
    widgets.HBox([search_input, search_button]),
    widgets.HBox([category_dropdown, top_k_slider]),
    output_area
], layout=widgets.Layout(padding='20px', border='1px solid #E5E7EB', border_radius='12px', background_color='#FFFFFF'))

display(ui_box)